In [2]:
from mibici.pipeline import load_stage
from mibici.data_preparation.quality_fixes import (
    remove_duplicate_trips,
    replace_invalid_birth_years,
)
import pandas as pd
data = load_stage("structure")

2026-10-17 06:02:20.880 | INFO     | mibici.config:<module>:11 - PROJ_ROOT path is: /home/jovyan/notebooks/mibici/mibici_predictions


### 1.- Missing Values

In [3]:
trips = data["viajes"].copy()
stations = data["estaciones"].copy()

display(trips.isna().sum())
display(stations.isna().sum())

Viaje_Id                0
Usuario_Id              0
Genero               5627
Año_de_nacimiento      60
Inicio_del_viaje        0
Fin_del_viaje           0
Origen_Id               0
Destino_Id              0
dtype: int64

id            0
name          0
latitude      0
longitude     0
dpcapacity    0
dtype: int64

### 2.- Duplicates

In [4]:
before = data["viajes"].copy()
after = remove_duplicate_trips(before)

assert len(after) == len(before.drop_duplicates())
assert after.duplicated().sum() == 0
assert after.columns.equals(before.columns)

print(f"Copias eliminadas: {len(before) - len(after):,}")

Copias eliminadas: 758


### 3.- Domain Constraints

In [5]:
from mibici.config import MIN_AGE, MAX_AGE

before = remove_duplicate_trips(data["viajes"].copy())

after = replace_invalid_birth_years(
    before,
    min_age=MIN_AGE,
    max_age=MAX_AGE,
)

age = (
    after["Inicio_del_viaje"].dt.year
    - after["Año_de_nacimiento"]
)

assert len(after) == len(before)
assert str(after["Año_de_nacimiento"].dtype) == "Int64"
assert age.dropna().between(MIN_AGE, MAX_AGE).all()

other_columns = before.columns.drop("Año_de_nacimiento")
assert after[other_columns].equals(before[other_columns])

print(f"Años faltantes: {after['Año_de_nacimiento'].isna().sum():,}")

Años faltantes: 83


### 4.- Continuous Variables

In [6]:
display(stations.loc[stations["dpcapacity"].le(0)])

,id,name,latitude,longitude,dpcapacity
94,99,ex-(ZPN-009) C. Del Corral / C. Nicolás Bravo,20.685108,-103.333746,0
104,109,ex-(ZPN-019) C. Cabecera Mpal. Juan Manuel / C.,20.723221,-103.385583,0
111,116,ex-(ZPN-026) C. 16 Sep. / C. Fray A. de Segovia,20.686920,-103.334670,0
129,138,ex-(ZPN-044),20.697050,-103.363220,0


### 5.- Inconsistency

In [7]:
assert trips["Origen_Id"].isin(stations["id"]).all()
assert trips["Destino_Id"].isin(stations["id"]).all()

### 6.- Temporal Relationships

In [8]:
duration = trips["Fin_del_viaje"] - trips["Inicio_del_viaje"]

assert duration.notna().all()
assert duration.ge(pd.Timedelta(0)).all()

print(f"Duraciones cero: {duration.eq(pd.Timedelta(0)).sum():,}")

Duraciones cero: 258


### 7.- Constant / Near-Constant Variables

In [9]:
data = load_stage("quality_fixes")

trips = data["viajes"]
stations = data["estaciones"]

def summarize_constants(df):
    return pd.DataFrame({
        "unique_count": df.nunique(dropna=False),
        "top_value_pct": [
            df[column]
            .value_counts(normalize=True, dropna=False)
            .max() * 100
            for column in df.columns
        ],
    }).round(2)


display(summarize_constants(trips))
display(summarize_constants(stations))

,unique_count,top_value_pct
Viaje_Id,882822,0.00
Usuario_Id,20169,0.09
Genero,3,72.06
Año_de_nacimiento,71,5.00
Inicio_del_viaje,770890,0.00
Fin_del_viaje,772051,0.00
Origen_Id,469,2.32
Destino_Id,469,2.56


,unique_count,top_value_pct
id,484,0.21
name,484,0.21
latitude,480,0.41
longitude,481,0.41
dpcapacity,9,41.53
